# Replay Token Cost Accounting

Purpose: compute token usage and estimated API cost for the study-round replay experiments.

Inputs:
- Combined replay CSV data from `load_replay_data()`.
- Per-model pricing from `shared.model_catalog.get_pricing_per_million()`.

Unit of analysis:
- One replay row per `(game_id, player_id, turn, condition, replay_model, repetition)`.

Definitions:
- `Combined Output` is `reasoning_tokens + output_tokens`.
- `Total Cost` applies input and output prices per million tokens to observed token counts.


In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from IPython.display import display

from shared.plot_utilities import setup_notebook_display
from shared.model_catalog import get_pricing_per_million
from nuke.utils.load_replay_data import (
    add_canonical_replay_model,
    load_replay_data,
)

setup_notebook_display()
df = load_replay_data()
df = add_canonical_replay_model(df)
df["combined_output_tokens"] = df["reasoning_tokens"] + df["output_tokens"]


print(f"Token columns present: input_tokens, reasoning_tokens, output_tokens")
print(f"Rows with any NaN tokens: {df[['input_tokens', 'reasoning_tokens', 'output_tokens']].isna().any(axis=1).sum()}")
print(f"Rows with zero input_tokens: {(df['input_tokens'] == 0).sum()}")


✓ Loaded 40,560 rows from 104 files
  Conditions   : original, no-rationale, high-stakes, high-stakes-no-rationale, ethical, ethical-high-stakes, ethical-no-rationale, high-stakes-no-rationale-ethical
  Replay models: DeepSeek-V3.2, DeepSeek-V4, GLM-4.7, GLM-5.1, Gemini-3.5-Flash, Gemma-4, Kimi-K2.5, Kimi-K2.6, MiniMax-M2.7, Mistral-Small-4, Qwen-3.5, Qwen-3.6-27B, gpt-oss-120b

  Rows per condition × replay model:
  ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                         DeepSeek-V3.2       DeepSeek-V4           GLM-4.7           GLM-5.1  Gemini-3.5-Flash           Gemma-4         Kimi-K2.5         Kimi-K2.6      MiniMax-M2.7   Mistral-Small-4          Qwen-3.5      Qwen-3.6-27B      gpt-oss-120b             Total
  ─

In [2]:
PRICING_PER_MILLION = get_pricing_per_million()
PRICING_PER_MILLION

{'GPT-OSS-120B': {'input_per_million': 0.039, 'output_per_million': 0.19},
 'Sonnet-4.5': {'input_per_million': 3.0, 'output_per_million': 15.0},
 'GLM-4.7': {'input_per_million': 0.39, 'output_per_million': 1.75},
 'GLM-5.1': {'input_per_million': 1.05, 'output_per_million': 3.5},
 'Minimax-M2.5': {'input_per_million': 0.2, 'output_per_million': 1.17},
 'Minimax-M2.7': {'input_per_million': 0.3, 'output_per_million': 1.2},
 'Kimi-K2.5': {'input_per_million': 0.4, 'output_per_million': 2},
 'Kimi-K2.6': {'input_per_million': 0.75, 'output_per_million': 3.5},
 'DeepSeek-3.2': {'input_per_million': 0.26, 'output_per_million': 0.38},
 'DeepSeek-4': {'input_per_million': 0.435, 'output_per_million': 0.87},
 'Qwen-3.5': {'input_per_million': 0.39, 'output_per_million': 2.34},
 'Qwen-3.6-27B': {'input_per_million': 0.13, 'output_per_million': 0.76},
 'Mistral-Small-4': {'input_per_million': 0.15, 'output_per_million': 0.6},
 'Gemma-4': {'input_per_million': 0.13, 'output_per_million': 0.38},

In [3]:
def format_number(value):
    if pd.isna(value):
        return 'N/A'
    return f'{value:,.0f}'


def format_currency(value):
    if pd.isna(value):
        return 'N/A'
    return f'${value:,.2f}'


def format_cost_per_unit(model_name, pricing_per_million):
    p = pricing_per_million.get(model_name, {})
    ip, op = p.get('input_per_million'), p.get('output_per_million')
    if ip is None or op is None:
        return 'Missing pricing'
    return f"${ip:,.2f} in / ${op:,.2f} out per 1M"


# Compute per-row cost
df['input_price_per_million'] = df['replay_model_canonical'].map(
    lambda m: PRICING_PER_MILLION.get(m, {}).get('input_per_million')
)
df['output_price_per_million'] = df['replay_model_canonical'].map(
    lambda m: PRICING_PER_MILLION.get(m, {}).get('output_per_million')
)
df['row_cost'] = (
    df['input_tokens'] / 1_000_000 * df['input_price_per_million']
    + df['combined_output_tokens'] / 1_000_000 * df['output_price_per_million']
)

# Summarize by condition and replay model
summary = (
    df.groupby(['condition', 'replay_model_canonical'], observed=True)
    .agg(
        Rows=('row_cost', 'size'),
        Total_Input=('input_tokens', 'sum'),
        Avg_Input=('input_tokens', 'mean'),
        Total_Output=('combined_output_tokens', 'sum'),
        Avg_Output=('combined_output_tokens', 'mean'),
        Total_Cost=('row_cost', 'sum'),
    )
    .reset_index()
    .rename(columns={
        'condition': 'Condition',
        'replay_model_canonical': 'Replay Model',
        'Total_Input': 'Total Input',
        'Avg_Input': 'Avg. Input',
        'Total_Output': 'Total Output',
        'Avg_Output': 'Avg. Output',
        'Total_Cost': 'Total Cost',
    })
)

summary['Cost Per Unit'] = summary['Replay Model'].map(
    lambda m: format_cost_per_unit(m, PRICING_PER_MILLION)
)

# Display formatted
summary_display = summary.copy()
for col in ['Rows', 'Total Input', 'Avg. Input', 'Total Output', 'Avg. Output']:
    summary_display[col] = summary_display[col].map(format_number)
summary_display['Total Cost'] = summary_display['Total Cost'].map(format_currency)

display(summary_display)
print(summary_display.to_markdown(index=False))

,Condition,Replay Model,Rows,Total Input,Avg. Input,Total Output,Avg. Output,Total Cost,Cost Per Unit
0,original,DeepSeek-3.2,390,"20,879,003","53,536","885,504","2,271",$5.77,$0.26 in / $0.38 out per 1M
1,original,DeepSeek-4,390,"21,998,413","56,406","1,360,157","3,488",$10.75,$0.43 in / $0.87 out per 1M
2,original,GLM-4.7,390,"21,306,145","54,631","745,496","1,912",$9.61,$0.39 in / $1.75 out per 1M
3,original,GLM-5.1,390,"21,179,872","54,307","2,261,041","5,798",$30.15,$1.05 in / $3.50 out per 1M
4,original,GPT-OSS-120B,390,"21,084,339","54,062","568,347","1,457",$0.93,$0.04 in / $0.19 out per 1M
...,...,...,...,...,...,...,...,...,...
99,high-stakes-no-rationale-ethical,Kimi-K2.6,390,"22,349,164","57,306","4,168,846","10,689",$31.35,$0.75 in / $3.50 out per 1M
100,high-stakes-no-rationale-ethical,Minimax-M2.7,390,"29,147,132","74,736","540,071","1,385",$9.39,$0.30 in / $1.20 out per 1M
101,high-stakes-no-rationale-ethical,Mistral-Small-4,390,"20,496,619","52,555","770,283","1,975",$3.54,$0.15 in / $0.60 out per 1M
102,high-stakes-no-rationale-ethical,Qwen-3.5,390,"25,662,733","65,802","820,672","2,104",$11.93,$0.39 in / $2.34 out per 1M


| Condition                        | Replay Model     |   Rows | Total Input   | Avg. Input   | Total Output   | Avg. Output   | Total Cost   | Cost Per Unit               |
|:---------------------------------|:-----------------|-------:|:--------------|:-------------|:---------------|:--------------|:-------------|:----------------------------|
| original                         | DeepSeek-3.2     |    390 | 20,879,003    | 53,536       | 885,504        | 2,271         | $5.77        | $0.26 in / $0.38 out per 1M |
| original                         | DeepSeek-4       |    390 | 21,998,413    | 56,406       | 1,360,157      | 3,488         | $10.75       | $0.43 in / $0.87 out per 1M |
| original                         | GLM-4.7          |    390 | 21,306,145    | 54,631       | 745,496        | 1,912         | $9.61        | $0.39 in / $1.75 out per 1M |
| original                         | GLM-5.1          |    390 | 21,179,872    | 54,307       | 2,261,041      | 5,798         | $

In [4]:
# Summary by replay model (collapsed across conditions)
model_summary = (
    df.groupby('replay_model_canonical', observed=True)
    .agg(
        Rows=('row_cost', 'size'),
        Conditions=('condition', 'nunique'),
        Total_Input=('input_tokens', 'sum'),
        Avg_Input=('input_tokens', 'mean'),
        Total_Output=('combined_output_tokens', 'sum'),
        Avg_Output=('combined_output_tokens', 'mean'),
        Total_Cost=('row_cost', 'sum'),
    )
    .reset_index()
    .rename(columns={
        'replay_model_canonical': 'Replay Model',
        'Total_Input': 'Total Input',
        'Avg_Input': 'Avg. Input',
        'Total_Output': 'Total Output',
        'Avg_Output': 'Avg. Output',
        'Total_Cost': 'Total Cost',
    })
)

model_summary['Cost Per Unit'] = model_summary['Replay Model'].map(
    lambda m: format_cost_per_unit(m, PRICING_PER_MILLION)
)

model_summary_display = model_summary.copy()
for col in ['Rows', 'Conditions', 'Total Input', 'Avg. Input', 'Total Output', 'Avg. Output']:
    model_summary_display[col] = model_summary_display[col].map(format_number)
model_summary_display['Total Cost'] = model_summary_display['Total Cost'].map(format_currency)

display(model_summary_display)
print(model_summary_display.to_markdown(index=False))

overall_total = summary['Total Cost'].sum()
print(f"\nOverall Total Cost: {format_currency(overall_total)}")

,Replay Model,Rows,Conditions,Total Input,Avg. Input,Total Output,Avg. Output,Total Cost,Cost Per Unit
0,DeepSeek-3.2,"3,120",8,"166,518,147","53,371","7,530,246","2,414",$46.16,$0.26 in / $0.38 out per 1M
1,DeepSeek-4,"3,120",8,"170,004,241","54,489","10,590,303","3,394",$83.17,$0.43 in / $0.87 out per 1M
2,GLM-4.7,"3,120",8,"155,516,821","49,845","5,247,933","1,682",$69.84,$0.39 in / $1.75 out per 1M
3,GLM-5.1,"3,120",8,"168,778,913","54,096","16,040,157","5,141",$233.36,$1.05 in / $3.50 out per 1M
4,GPT-OSS-120B,"3,120",8,"166,422,828","53,341","4,984,539","1,598",$7.44,$0.04 in / $0.19 out per 1M
5,Gemini-3.5-Flash,"3,120",8,"199,545,956","63,957","24,013,984","7,697",$257.72,$0.75 in / $4.50 out per 1M
6,Gemma-4,"3,120",8,"171,799,392","55,064","5,309,942","1,702",$24.35,$0.13 in / $0.38 out per 1M
7,Kimi-K2.5,"3,120",8,"165,463,993","53,033","10,474,897","3,357",$87.14,$0.40 in / $2.00 out per 1M
8,Kimi-K2.6,"3,120",8,"174,260,493","55,853","34,257,284","10,980",$250.60,$0.75 in / $3.50 out per 1M
9,Minimax-M2.7,"3,120",8,"209,630,198","67,189","4,532,990","1,453",$68.33,$0.30 in / $1.20 out per 1M


| Replay Model     | Rows   |   Conditions | Total Input   | Avg. Input   | Total Output   | Avg. Output   | Total Cost   | Cost Per Unit               |
|:-----------------|:-------|-------------:|:--------------|:-------------|:---------------|:--------------|:-------------|:----------------------------|
| DeepSeek-3.2     | 3,120  |            8 | 166,518,147   | 53,371       | 7,530,246      | 2,414         | $46.16       | $0.26 in / $0.38 out per 1M |
| DeepSeek-4       | 3,120  |            8 | 170,004,241   | 54,489       | 10,590,303     | 3,394         | $83.17       | $0.43 in / $0.87 out per 1M |
| GLM-4.7          | 3,120  |            8 | 155,516,821   | 49,845       | 5,247,933      | 1,682         | $69.84       | $0.39 in / $1.75 out per 1M |
| GLM-5.1          | 3,120  |            8 | 168,778,913   | 54,096       | 16,040,157     | 5,141         | $233.36      | $1.05 in / $3.50 out per 1M |
| GPT-OSS-120B     | 3,120  |            8 | 166,422,828   | 53,341       | 